# LLaMA-3.1-8B Vanilla BFP Zero-Shot Accuracy

Evaluate BFP8 through BFP4 with the same eight-task zero-shot protocol as the FP16 baseline: LAMBADA, ARC-Easy, ARC-Challenge, PIQA, HellaSwag, WinoGrande, BoolQ, and COPA. Each format reloads the pinned FP16 checkpoint, quantizes decoder `nn.Linear` weights and activations with G16 round-to-nearest-even BFP, and leaves `lm_head` in FP16.

The notebook uses `lm-evaluation-harness==0.4.13`, evaluates all 25,218 examples for every format, logs complete samples, checks evaluation closure, reports deltas from the matching FP16 baseline JSON when it is uploaded, and stores per-layer weight, activation, and `weight_exp + activation_exp` histograms.

In [ ]:
%pip install -q "lm_eval[hf]==0.4.13" sentencepiece

In [ ]:
import gc
import importlib.metadata
import json
import os
import platform
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import datasets
import lm_eval
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import model_info
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager
from lm_eval.utils import make_table
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B"
TASKS = ["lambada_openai", "arc_easy", "arc_challenge", "piqa", "hellaswag", "winogrande", "boolq", "copa"]
NUM_FEWSHOT = 0
BATCH_SIZE = "auto"
MAX_BATCH_SIZE = 64
BOOTSTRAP_ITERS = 0
LOG_SAMPLES = True
MANTISSA_BITS_SWEEP = (7, 6, 5, 4, 3)  # BFP8, BFP7, BFP6, BFP5, BFP4.
OUTPUT_DIR = Path("result")
ARCHIVE_PATH = Path("llama3.1-8b-bfp8-bfp4-g16-rne-no-lm-head-zero-shot.zip")

EXPECTED_SAMPLES = {
    "lambada_openai": 5153,
    "arc_easy": 2376,
    "arc_challenge": 1172,
    "piqa": 1838,
    "hellaswag": 10042,
    "winogrande": 1267,
    "boolq": 3270,
    "copa": 100,
}
PAPER_METRICS = {
    "lambada_openai": "acc,none",
    "arc_easy": "acc_norm,none",
    "arc_challenge": "acc_norm,none",
    "piqa": "acc_norm,none",
    "hellaswag": "acc_norm,none",
    "winogrande": "acc,none",
    "boolq": "acc,none",
    "copa": "acc,none",
}
DISPLAY_NAMES = {
    "lambada_openai": "LAMBADA",
    "arc_easy": "ARC-Easy",
    "arc_challenge": "ARC-Challenge",
    "piqa": "PIQA",
    "hellaswag": "HellaSwag",
    "winogrande": "WinoGrande",
    "boolq": "BoolQ",
    "copa": "COPA",
}
MODEL_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
# Upload the matching 01_Baseline result JSON next to this notebook to record deltas vs FP16.
FP16_BASELINE_PATH = Path("llama3.1-8b-fp16-zero-shot.json")

@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 7
    rounding: str = "nearest_even"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This experiment is fixed to G16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This experiment requires a 5-bit shared exponent.")
        if not 1 <= self.mantissa_bits <= 11:
            raise ValueError("mantissa_bits must be between 1 and 11.")
        if self.rounding != "nearest_even":
            raise ValueError("This experiment requires round-to-nearest-even.")

BFP = BFPConfig()
BFP.validate()
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")
if importlib.metadata.version("lm_eval") != "0.4.13":
    raise RuntimeError("This experiment requires lm_eval==0.4.13.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(1234)
torch.backends.cuda.matmul.allow_tf32 = False

## BFP quantization and telemetry

Each Linear K dimension is partitioned into contiguous groups of 16. The shared exponent is the maximum unbiased FP16 exponent in the group. Mantissas use round-to-nearest-even and symmetric saturation. BFP8 is `1S7M`; the other formats reduce only the stored mantissa width. Automatic batch-size probes are quantized but excluded from runtime histograms.

In [ ]:
def _quantize_bfp_rows(rows, config, return_metadata=False):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    raw_shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    zero_mask = max_abs == 0
    low_clamped = (~zero_mask) & (raw_shared_exp < exp_min)
    high_clamped = (~zero_mask) & (raw_shared_exp > exp_max)
    shared_exp = raw_shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(zero_mask, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(-mantissa_max, mantissa_max)
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)
    if not return_metadata:
        return dequantized
    return dequantized, {
        "shared_exp": shared_exp.squeeze(-1).to(torch.int16),
        "zero_mask": zero_mask.squeeze(-1),
        "low_clamped": low_clamped.squeeze(-1),
        "high_clamped": high_clamped.squeeze(-1),
    }


def quantize_bfp(tensor, config, chunk_rows, return_metadata=False):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config, return_metadata)

    output = torch.empty_like(flat)
    metadata_parts = ({
        "shared_exp": [],
        "zero_mask": [],
        "low_clamped": [],
        "high_clamped": [],
    } if return_metadata else None)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        if return_metadata:
            quantized, metadata = _quantize_bfp_rows(
                flat[start:end], config, return_metadata=True
            )
            output[start:end] = quantized
            for key in metadata_parts:
                metadata_parts[key].append(metadata[key])
        else:
            output[start:end] = _quantize_bfp_rows(flat[start:end], config)
    output = output.reshape_as(tensor)
    if not return_metadata:
        return output
    return output, {key: torch.cat(parts, dim=0) for key, parts in metadata_parts.items()}


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    metadata_parts = {
        "shared_exp": [],
        "zero_mask": [],
        "low_clamped": [],
        "high_clamped": [],
    }
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        quantized, metadata = _quantize_bfp_rows(
            weight[start:end], config, return_metadata=True
        )
        weight[start:end].copy_(quantized)
        for key in metadata_parts:
            metadata_parts[key].append(metadata[key])
    return {key: torch.cat(parts, dim=0) for key, parts in metadata_parts.items()}


def _group_exponent_histogram(shared_exp, zero_mask, exp_min, num_bins):
    rows, groups = shared_exp.shape
    group_offsets = torch.arange(
        groups, device=shared_exp.device, dtype=torch.long
    ) * num_bins
    indices = shared_exp.long() - exp_min + group_offsets.unsqueeze(0)
    indices = indices[~zero_mask]
    return torch.bincount(
        indices.reshape(-1), minlength=groups * num_bins
    ).reshape(groups, num_bins)


def _histogram_record(counts, bins, zero_blocks, low_clamped, high_clamped):
    nonzero_bins = torch.nonzero(counts, as_tuple=False).flatten()
    minimum = None if nonzero_bins.numel() == 0 else int(bins[nonzero_bins[0]].item())
    maximum = None if nonzero_bins.numel() == 0 else int(bins[nonzero_bins[-1]].item())
    return {
        "bins": bins.detach().cpu().tolist(),
        "counts": counts.detach().cpu().tolist(),
        "nonzero_blocks": int(counts.sum().item()),
        "total_blocks": int(counts.sum().item() + zero_blocks.item()),
        "minimum": minimum,
        "maximum": maximum,
        "zero_blocks": int(zero_blocks.item()),
        "low_clamped_blocks": int(low_clamped.item()),
        "high_clamped_blocks": int(high_clamped.item()),
    }


class BFPLinear(nn.Module):
    def __init__(self, linear, config, weight_metadata):
        super().__init__()
        self.linear = linear
        self.config = config
        self.collect_telemetry = True
        self.exp_min = -(1 << (config.shared_exponent_bits - 1))
        self.exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
        self.num_exp_bins = self.exp_max - self.exp_min + 1
        weight_group_counts = _group_exponent_histogram(
            weight_metadata["shared_exp"], weight_metadata["zero_mask"],
            self.exp_min, self.num_exp_bins
        )
        self.register_buffer(
            "weight_group_counts_f64", weight_group_counts.to(torch.float64), persistent=False
        )
        self.register_buffer(
            "weight_shared_exp_counts", weight_group_counts.sum(dim=0), persistent=False
        )
        self.register_buffer(
            "activation_shared_exp_counts",
            torch.zeros(self.num_exp_bins, dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        self.register_buffer(
            "product_shared_exp_counts",
            torch.zeros(2 * self.num_exp_bins - 1, dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        self.register_buffer(
            "product_zero_partial_bases",
            torch.zeros((), dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        self.register_buffer("weight_zero_blocks", weight_metadata["zero_mask"].sum(dtype=torch.int64), persistent=False)
        self.register_buffer("weight_low_clamped_blocks", weight_metadata["low_clamped"].sum(dtype=torch.int64), persistent=False)
        self.register_buffer("weight_high_clamped_blocks", weight_metadata["high_clamped"].sum(dtype=torch.int64), persistent=False)
        for name in ("zero_blocks", "low_clamped_blocks", "high_clamped_blocks"):
            self.register_buffer(
                f"activation_{name}",
                torch.zeros((), dtype=torch.int64, device=linear.weight.device),
                persistent=False,
            )

    @torch.no_grad()
    def reset_runtime_telemetry(self):
        self.activation_shared_exp_counts.zero_()
        self.product_shared_exp_counts.zero_()
        self.product_zero_partial_bases.zero_()
        self.activation_zero_blocks.zero_()
        self.activation_low_clamped_blocks.zero_()
        self.activation_high_clamped_blocks.zero_()

    @torch.no_grad()
    def _update_exponent_histograms(self, metadata):
        activation_group_counts = _group_exponent_histogram(
            metadata["shared_exp"], metadata["zero_mask"],
            self.exp_min, self.num_exp_bins
        )
        self.activation_shared_exp_counts.add_(activation_group_counts.sum(dim=0))
        self.activation_zero_blocks.add_(metadata["zero_mask"].sum(dtype=torch.int64))
        self.activation_low_clamped_blocks.add_(metadata["low_clamped"].sum(dtype=torch.int64))
        self.activation_high_clamped_blocks.add_(metadata["high_clamped"].sum(dtype=torch.int64))
        pair_counts = (
            activation_group_counts.transpose(0, 1).to(torch.float64)
            @ self.weight_group_counts_f64
        ).round().to(torch.int64)
        total_partial_bases = metadata["shared_exp"].numel() * self.linear.out_features
        self.product_zero_partial_bases.add_(total_partial_bases - pair_counts.sum())
        for activation_index in range(self.num_exp_bins):
            self.product_shared_exp_counts[
                activation_index : activation_index + self.num_exp_bins
            ].add_(pair_counts[activation_index])

    def export_exponent_histograms(self):
        shared_bins = torch.arange(
            self.exp_min, self.exp_max + 1, device=self.linear.weight.device
        )
        product_bins = torch.arange(
            2 * self.exp_min, 2 * self.exp_max + 1, device=self.linear.weight.device
        )
        weight = _histogram_record(
            self.weight_shared_exp_counts, shared_bins, self.weight_zero_blocks,
            self.weight_low_clamped_blocks, self.weight_high_clamped_blocks,
        )
        activation = _histogram_record(
            self.activation_shared_exp_counts, shared_bins, self.activation_zero_blocks,
            self.activation_low_clamped_blocks, self.activation_high_clamped_blocks,
        )
        product = {
            "definition": "weight_shared_exp + activation_shared_exp",
            "bins": product_bins.detach().cpu().tolist(),
            "counts": self.product_shared_exp_counts.detach().cpu().tolist(),
            "nonzero_partial_bases": int(self.product_shared_exp_counts.sum().item()),
            "zero_partial_bases": int(self.product_zero_partial_bases.item()),
        }
        product["total_partial_bases"] = product["nonzero_partial_bases"] + product["zero_partial_bases"]
        expected_product_total = activation["total_blocks"] * self.linear.out_features
        if product["total_partial_bases"] != expected_product_total:
            raise RuntimeError("Product shared-exponent histogram closure failed.")
        return {
            "input_features": self.linear.in_features,
            "output_features": self.linear.out_features,
            "groups_per_row": self.weight_group_counts_f64.size(0),
            "weight_shared_exponent": weight,
            "activation_shared_exponent": activation,
            "product_shared_exponent": product,
        }

    def forward(self, x):
        if self.collect_telemetry:
            x_bfp, metadata = quantize_bfp(
                x, self.config, self.config.activation_chunk_rows, return_metadata=True
            )
            self._update_exponent_histograms(metadata)
        else:
            x_bfp = quantize_bfp(
                x, self.config, self.config.activation_chunk_rows, return_metadata=False
            )
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = {}
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            weight_metadata = quantize_weight_in_place(child.weight, config)
            wrapper = BFPLinear(child, config, weight_metadata)
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(replace_linear_layers(child, config, full_name))
    return replaced


def export_layer_exponent_histograms(layers):
    return [
        {"layer_name": name, **layer.export_exponent_histograms()}
        for name, layer in layers.items()
    ]


class BFPEvaluationHFLM(HFLM):
    def __init__(self, *args, bfp_layers, **kwargs):
        self.bfp_layers = bfp_layers
        super().__init__(*args, **kwargs)

    def _detect_batch_size(self, requests=None, pos=0):
        for layer in self.bfp_layers.values():
            layer.collect_telemetry = False
        try:
            return super()._detect_batch_size(requests=requests, pos=pos)
        finally:
            for layer in self.bfp_layers.values():
                layer.reset_runtime_telemetry()
                layer.collect_telemetry = True


sample = torch.tensor(
    [[1.0078125, 1.0234375, -1.0234375, 1.5]],
    device="cuda", dtype=torch.float16,
)
expected_q = torch.tensor(
    [[1.0, 1.03125, -1.03125, 1.5]],
    device="cuda", dtype=torch.float16,
)
assert torch.equal(_quantize_bfp_rows(sample, BFP), expected_q)

## Validate the shared evaluation protocol

The model revision must match the pinned revision and every task count must close. Upload the matching `01_Baseline` result JSON (`FP16_BASELINE_PATH`) to the Colab working directory to record deltas vs FP16; the notebook checks that its model, revision, lm-eval version, tasks, metrics, and example count match. Without it, deltas are stored as null. Request caching is deliberately disabled because cached model responses would bypass BFP forwards and invalidate activation telemetry.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")

task_manager = TaskManager()
matched_tasks = task_manager.match_tasks(TASKS)
missing_tasks = sorted(set(TASKS) - set(matched_tasks))
if missing_tasks:
    raise RuntimeError(f"Tasks missing from lm-eval: {missing_tasks}")

resolved_model_revision = model_info(MODEL_ID, token=token).sha
if resolved_model_revision != MODEL_REVISION:
    raise RuntimeError(
        "Model revision differs from the pinned revision: "
        f"{resolved_model_revision} != {MODEL_REVISION}"
    )
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=resolved_model_revision, token=token
)
print(f"lm-eval: {importlib.metadata.version('lm_eval')}")
print(f"Model revision: {resolved_model_revision}")
print(f"Tasks: {', '.join(TASKS)}")
print(f"Examples per format: {sum(EXPECTED_SAMPLES.values()):,}")
print(f"Formats: {[f'BFP{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")


def load_fp16_baseline(path):
    if not path.is_file():
        print(f"FP16 baseline {path} not found; deltas vs FP16 will be stored as null.")
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    checks = {
        "model": (data.get("model"), MODEL_ID),
        "model_revision": (data.get("model_revision"), MODEL_REVISION),
        "num_fewshot": (data.get("num_fewshot"), NUM_FEWSHOT),
        "lm_eval": (data.get("environment", {}).get("lm_eval"), importlib.metadata.version("lm_eval")),
        "tasks": (list(data.get("paper_scores", {})), TASKS),
        "metrics": (
            {task: row["metric"] for task, row in data.get("paper_scores", {}).items()},
            PAPER_METRICS,
        ),
        "total_evaluated_examples": (
            data.get("total_evaluated_examples"), sum(EXPECTED_SAMPLES.values())
        ),
    }
    for name, (found, expected) in checks.items():
        if found != expected:
            raise RuntimeError(f"FP16 baseline {name} mismatch: {found!r} != {expected!r}")
    scores = {task: float(data["paper_scores"][task]["score"]) for task in TASKS}
    return {
        "source": path.name,
        "created_at_utc": data.get("created_at_utc"),
        "model_revision": data["model_revision"],
        "lm_eval": data["environment"]["lm_eval"],
        "num_fewshot": data["num_fewshot"],
        "total_evaluated_examples": data["total_evaluated_examples"],
        "scores": scores,
        "macro_average": sum(scores.values()) / len(TASKS),
    }


def format_macro_delta(result):
    delta = result["macro_delta_percentage_points_vs_fp16"]
    return "n/a (no FP16 baseline JSON)" if delta is None else f"{delta:+.2f} pp"


BASELINE_REFERENCE = load_fp16_baseline(FP16_BASELINE_PATH)
if BASELINE_REFERENCE is not None:
    print(
        f"FP16 baseline: {BASELINE_REFERENCE['source']} "
        f"(macro average {BASELINE_REFERENCE['macro_average'] * 100:.2f}%)"
    )


In [ ]:
def to_jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def summarize_evaluation(raw_results):
    closure = {}
    for task_name, expected in EXPECTED_SAMPLES.items():
        sample_info = raw_results["n-samples"][task_name]
        original = int(sample_info["original"])
        effective = int(sample_info["effective"])
        passed = original == expected and effective == expected
        closure[task_name] = {
            "expected": expected,
            "original": original,
            "effective": effective,
            "passed": passed,
        }
        if not passed:
            raise RuntimeError(
                f"Evaluation closure failed for {task_name}: "
                f"expected={expected}, original={original}, effective={effective}"
            )

    paper_scores = {}
    for task_name in TASKS:
        metric_name = PAPER_METRICS[task_name]
        metrics = raw_results["results"][task_name]
        if metric_name not in metrics:
            raise KeyError(
                f"Missing {metric_name!r} for {task_name}; available: {sorted(metrics)}"
            )
        score = float(metrics[metric_name])
        baseline_score = (
            None if BASELINE_REFERENCE is None
            else BASELINE_REFERENCE["scores"][task_name]
        )
        stderr_name = metric_name.replace(",none", "_stderr,none")
        stderr_raw = metrics.get(stderr_name)
        try:
            stderr = float(stderr_raw)
        except (TypeError, ValueError):
            stderr = None
        paper_scores[task_name] = {
            "display_name": DISPLAY_NAMES[task_name],
            "metric": metric_name,
            "score": score,
            "score_percent": score * 100.0,
            "fp16_baseline_score": baseline_score,
            "fp16_baseline_score_percent": None if baseline_score is None else baseline_score * 100.0,
            "delta_vs_fp16": None if baseline_score is None else score - baseline_score,
            "delta_percentage_points_vs_fp16": (
                None if baseline_score is None else (score - baseline_score) * 100.0
            ),
            "stderr": stderr,
            "stderr_percent": None if stderr is None else stderr * 100.0,
        }

    macro_average = sum(x["score"] for x in paper_scores.values()) / len(TASKS)
    return closure, paper_scores, macro_average


## Run BFP8 through BFP4

Each result is written immediately after its format finishes, so completed formats remain available if a later run is interrupted.

In [ ]:
results = []
output_paths = []

for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(BFP, mantissa_bits=mantissa_bits)
    config.validate()
    bfp_bits = 1 + config.mantissa_bits
    output_path = OUTPUT_DIR / f"bfp{bfp_bits}-g16-rne-no-lm-head-zero-shot.json"

    print(f"\n{'=' * 72}")
    print(f"Running BFP{bfp_bits}: {config}")
    print('=' * 72)

    torch.manual_seed(1234)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=resolved_model_revision,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False
    quantized_layers = replace_linear_layers(model, config)
    quantized_layer_count = len(quantized_layers)
    torch.cuda.empty_cache()

    parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
    parameter_devices = {p.device.type for p in model.parameters()}
    assert parameter_dtypes == {torch.float16}, parameter_dtypes
    assert parameter_devices == {"cuda"}, parameter_devices
    assert quantized_layer_count == 224, quantized_layer_count
    assert config.block_size == 16
    assert config.quantize_lm_head is False
    assert isinstance(model.lm_head, nn.Linear)

    eval_model = BFPEvaluationHFLM(
        pretrained=model,
        tokenizer=tokenizer,
        batch_size=BATCH_SIZE,
        max_batch_size=MAX_BATCH_SIZE,
        bfp_layers=quantized_layers,
    )
    evaluation_start = time.perf_counter()
    raw_results = lm_eval.simple_evaluate(
        model=eval_model,
        tasks=TASKS,
        num_fewshot=NUM_FEWSHOT,
        batch_size=BATCH_SIZE,
        max_batch_size=MAX_BATCH_SIZE,
        device="cuda:0",
        limit=None,
        use_cache=None,
        bootstrap_iters=BOOTSTRAP_ITERS,
        check_integrity=False,
        log_samples=LOG_SAMPLES,
        task_manager=task_manager,
        random_seed=0,
        numpy_random_seed=1234,
        torch_random_seed=1234,
        fewshot_random_seed=1234,
    )
    evaluation_elapsed_seconds = time.perf_counter() - evaluation_start
    if raw_results is None:
        raise RuntimeError("lm-eval returned no results.")

    closure, paper_scores, macro_average = summarize_evaluation(raw_results)
    layer_exponent_histograms = export_layer_exponent_histograms(quantized_layers)
    if len(layer_exponent_histograms) != quantized_layer_count:
        raise RuntimeError("Layer exponent metadata count mismatch.")
    if not all(
        layer["activation_shared_exponent"]["total_blocks"] > 0
        for layer in layer_exponent_histograms
    ):
        raise RuntimeError("One or more layers have empty activation telemetry.")

    raw_jsonable = json.loads(
        json.dumps(raw_results, ensure_ascii=False, default=to_jsonable)
    )
    macro_delta = (
        None if BASELINE_REFERENCE is None
        else macro_average - BASELINE_REFERENCE["macro_average"]
    )
    result = {
        "schema_version": 1,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment": "vanilla_bfp_zero_shot_accuracy",
        "model": MODEL_ID,
        "model_revision": resolved_model_revision,
        "dtype": "float16",
        "quantized": True,
        "quantization": "decoder Linear W/A BFP fake quantization; FP16 lm_head",
        "format": f"BFP{bfp_bits} (1S{config.mantissa_bits}M + shared E{config.shared_exponent_bits})",
        "bfp_config": asdict(config),
        "sweep_mantissa_bits": list(MANTISSA_BITS_SWEEP),
        "shared_exponent_contract": {
            "selection": "maximum unbiased exponent in each contiguous G16 block",
            "formula": "max(floor(log2(abs(x)))) over nonzero block values",
            "semantics": "MSFP maximum exponent, not mantissa-LSB scale exponent",
            "encoding": "signed integer",
            "zero_block_exponent": 0,
        },
        "mantissa_rounding_contract": "round-to-nearest-even (torch.round), followed by symmetric saturation",
        "quantized_scope": {
            "operations": "LLaMA decoder nn.Linear modules",
            "weights": True,
            "activations": True,
            "lm_head": False,
            "attention_internal_matmul": False,
            "softmax": False,
            "layernorm": False,
        },
        "linear_output_dtype": "float16",
        "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
        "quantized_linear_layers": quantized_layer_count,
        "attention_implementation": "eager",
        "tasks": TASKS,
        "num_fewshot": NUM_FEWSHOT,
        "limit": None,
        "batch_size": BATCH_SIZE,
        "max_batch_size": MAX_BATCH_SIZE,
        "bootstrap_iters": BOOTSTRAP_ITERS,
        "log_samples": LOG_SAMPLES,
        "request_cache_enabled": False,
        "paper_metric_policy": {
            "lambada": "acc",
            "arc_easy_arc_challenge_piqa_hellaswag": "acc_norm",
            "winogrande_boolq_copa": "acc",
        },
        "paper_scores": paper_scores,
        "macro_average": macro_average,
        "macro_average_percent": macro_average * 100.0,
        "fp16_baseline_reference": BASELINE_REFERENCE,
        "macro_delta_vs_fp16": macro_delta,
        "macro_delta_percentage_points_vs_fp16": None if macro_delta is None else macro_delta * 100.0,
        "evaluation_closure": closure,
        "total_evaluated_examples": sum(
            item["effective"] for item in closure.values()
        ),
        "evaluation_elapsed_seconds": evaluation_elapsed_seconds,
        "exponent_telemetry": {
            "storage": "per-layer exact histograms; raw exponent streams are not stored",
            "runtime_scope": "official evaluation forwards; auto-batch probes excluded",
            "product_shared_exponent_definition": "weight_shared_exp + activation_shared_exp",
            "actual_partial_sum_leading_exponent_profiled": False,
            "layers": layer_exponent_histograms,
        },
        "validation": {
            "expected_quantized_linear_layers": 224,
            "expected_evaluation_counts": EXPECTED_SAMPLES,
            "evaluation_closure_passed": True,
            "histogram_closure_passed": True,
            "auto_batch_probe_telemetry_excluded": True,
        },
        "environment": {
            "gpu": torch.cuda.get_device_name(0),
            "cuda": torch.version.cuda,
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "transformers": transformers.__version__,
            "datasets": datasets.__version__,
            "lm_eval": importlib.metadata.version("lm_eval"),
        },
        "lm_eval_raw": raw_jsonable,
    }

    output_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    results.append(result)
    output_paths.append(output_path)
    print(make_table(raw_results))
    print(f"BFP{bfp_bits} average: {result['macro_average_percent']:.2f}%")
    print(
        f"Delta vs FP16: {format_macro_delta(result)}"
    )
    print(f"Saved: {output_path.resolve()}")

    del eval_model, model, quantized_layers
    gc.collect()
    torch.cuda.empty_cache()

with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for output_path in output_paths:
        archive.write(output_path, arcname=output_path.name)

print("\nSweep complete:")
for result in results:
    print(
        f"{result['format'].split()[0]}: "
        f"average={result['macro_average_percent']:.2f}%, "
        f"delta={format_macro_delta(result)}"
    )
print(f"Archive: {ARCHIVE_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))